In [1]:
%pip install hf_xet

   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   - -------------------------------------- 0.1/2.8 MB 655.4 kB/s eta 0:00:05
   - -------------------------------------- 0.1/2.8 MB 726.2 kB/s eta 0:00:04
   -- ------------------------------------- 0.2/2.8 MB 748.1 kB/s eta 0:00:04
   --- ------------------------------------ 0.2/2.8 MB 885.4 kB/s eta 0:00:03
   --- ------------------------------------ 0.3/2.8 MB 850.6 kB/s eta 0:00:03
   ----- ---------------------------------- 0.4/2.8 MB 995.6 kB/s eta 0:00:03
   ------- -------------------------------- 0.5/2.8 MB 1.3 MB/s eta 0:00:02
   --------- ------------------------------ 0.7/2.8 MB 1.5 MB/s eta 0:00:02
   ---------- ----------------------------- 0.7/2.8 MB 1.5 MB/s eta 0:00:02
   ---------- ----------------------------- 0.7/2.8 MB 1.5 MB/s eta 0:00:02
   ---------- -------

Vector Database Setup (locally with Chroma)

Loads processed chunks, generates embeddings, and creates Chroma vector database.
- Uses sentence-transformers for embeddings
- Sets up Chroma with persistence
- Optimizes for retrieval performance
- Prepares for RAG pipeline

In [1]:
import os
import json
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
from pathlib import Path
from typing import Dict, List
from tqdm import tqdm
import numpy as np
import shutil

class EnhancedVectorDatabaseBuilder:
    
    def __init__(self, 
                 persist_directory: str = "D:/PsyWiz/vector_db",
                 embedding_model: str = "sentence-transformers/all-MiniLM-L6-v2",
                 collection_name: str = "psywiz_papers"):
        
        self.persist_directory = Path(persist_directory)
        self.collection_name = collection_name
        
        # Check for database compatibility issues and clean if needed
        self._handle_database_compatibility()
        
        print(f"Loading embedding model: {embedding_model}")
        self.embedding_model = SentenceTransformer(embedding_model)
        
        self.client = chromadb.PersistentClient(
            path=str(self.persist_directory),
            settings=Settings(
                anonymized_telemetry=False,
                allow_reset=True
            )
        )
        
        try:
            self.collection = self.client.get_collection(name=collection_name)
            print(f"Loaded existing collection: {collection_name}")
        except Exception as e:
            print(f"Could not load existing collection: {e}")
            try:
                self.collection = self.client.create_collection(
                    name=collection_name,
                    metadata={"description": "PsyWiz medical research papers chunks with enhanced metadata"}
                )
                print(f"Created new collection: {collection_name}")
            except Exception as create_error:
                print(f"Error creating collection: {create_error}")
                print("Attempting to reset database...")
                self._reset_database()
                self.collection = self.client.create_collection(
                    name=collection_name,
                    metadata={"description": "PsyWiz medical research papers chunks with enhanced metadata"}
                )
                print(f"Created new collection after reset: {collection_name}")
    
    def _handle_database_compatibility(self):
        """Handle database compatibility issues"""
        if self.persist_directory.exists():
            # Check for common database files that might be incompatible
            db_files = list(self.persist_directory.glob("*.sqlite*"))
            if db_files:
                print(f"Found existing database files: {[f.name for f in db_files]}")
                user_input = input("Database might be incompatible. Reset database? (y/N): ")
                if user_input.lower() == 'y':
                    self._reset_database()
        else:
            self.persist_directory.mkdir(parents=True, exist_ok=True)
    
    def _reset_database(self):
        """Reset the database directory"""
        if self.persist_directory.exists():
            print(f"Resetting database directory: {self.persist_directory}")
            shutil.rmtree(self.persist_directory)
        self.persist_directory.mkdir(parents=True, exist_ok=True)
        print("Database directory reset successfully")
    
    def load_chunks(self, chunks_file: str) -> List[Dict]:
        """Load processed chunks from JSON file"""
        print(f"Loading chunks from: {chunks_file}")
        
        with open(chunks_file, 'r', encoding='utf-8') as f:
            chunks = json.load(f)
        
        print(f"Loaded {len(chunks)} chunks")
        
        # Validate chunk structure
        if chunks:
            sample_chunk = chunks[0]
            print(f"Sample chunk metadata keys: {list(sample_chunk.get('metadata', {}).keys())}")
            
        return chunks
    
    def generate_embeddings(self, texts: List[str], batch_size: int = 32) -> np.ndarray:
        """Generate embeddings for text chunks"""
        print(f"Generating embeddings for {len(texts)} chunks...")
        
        # Process in batches for memory efficiency
        all_embeddings = []
        
        for i in tqdm(range(0, len(texts), batch_size), desc="Embedding batches"):
            batch_texts = texts[i:i + batch_size]
            batch_embeddings = self.embedding_model.encode(
                batch_texts,
                convert_to_numpy=True,
                show_progress_bar=False
            )
            all_embeddings.append(batch_embeddings)
        
        embeddings = np.vstack(all_embeddings)
        print(f"Generated embeddings shape: {embeddings.shape}")
        
        return embeddings
    
    def prepare_chunk_data(self, chunks: List[Dict]) -> tuple:
        """Prepare chunk data with ENHANCED metadata for ChromaDB"""
        chunk_ids = []
        documents = []
        metadatas = []
        
        for chunk in chunks:
            chunk_ids.append(chunk['chunk_id'])
            documents.append(chunk['content'])
            
            # ENHANCED: Extract all metadata fields from the improved chunking
            chunk_meta = chunk['metadata']
            
            metadata = {
                # Core identifiers
                'document_id': chunk_meta.get('document_id', ''),
                'chunk_id': chunk_meta.get('chunk_id', chunk['chunk_id']),
                'source_file': chunk_meta.get('source_file', ''),
                
                # Paper metadata (FIXED: Now includes all the rich data)
                'paper_title': chunk_meta.get('paper_title', 'Unknown Title'),
                'doi': chunk_meta.get('doi', ''),
                'publication_date': chunk_meta.get('publication_date', ''),
                'journal': chunk_meta.get('journal', ''),
                'source_url': chunk_meta.get('source_url', ''),
                
                # Authors (handle list properly)
                'authors': self._format_authors(chunk_meta.get('authors', [])),
                
                # Section information
                'section': chunk_meta.get('section', ''),
                'section_index': chunk_meta.get('section_index', 0),
                'chunk_index': chunk_meta.get('chunk_index', 0),
                
                # Content metadata
                'token_count': chunk.get('token_count', chunk_meta.get('token_count', 0)),
                'content_preview': chunk_meta.get('content_preview', chunk['content'][:100] + "..."),
                
                # Additional metadata
                'total_chunks': chunk_meta.get('total_chunks', 0),
                'priority_score': chunk_meta.get('priority_score', 0.7),
                
                # Abstract and keywords (if available)
                'abstract': chunk_meta.get('abstract', '')[:200] if chunk_meta.get('abstract') else '',
                'keywords': ', '.join(chunk_meta.get('keywords', [])) if chunk_meta.get('keywords') else ''
            }
            
            metadatas.append(metadata)
        
        return chunk_ids, documents, metadatas
    
    def _format_authors(self, authors) -> str:
        """Format authors list into a string for ChromaDB"""
        if isinstance(authors, list):
            return ', '.join(authors) if authors else ''
        elif isinstance(authors, str):
            return authors
        else:
            return ''
    
    def add_chunks_to_collection(self, chunks: List[Dict], batch_size: int = 100):
        """Add chunks to Chroma collection with enhanced embeddings and metadata"""
        
        existing_count = self.collection.count()
        if existing_count > 0:
            print(f"Collection already contains {existing_count} documents")
            user_input = input("Do you want to clear and rebuild? (y/N): ")
            if user_input.lower() == 'y':
                self.client.delete_collection(self.collection_name)
                self.collection = self.client.create_collection(
                    name=self.collection_name,
                    metadata={"description": "PsyWiz medical research papers chunks with enhanced metadata"}
                )
                print("Collection cleared and recreated")
            else:
                print("Skipping ingestion. Collection unchanged.")
                return
        
        # Prepare enhanced data
        chunk_ids, documents, metadatas = self.prepare_chunk_data(chunks)
        
        # Generate embeddings
        embeddings = self.generate_embeddings(documents)
        
        # Add to collection in batches
        print(f"Adding {len(chunks)} chunks to collection...")
        
        for i in tqdm(range(0, len(chunks), batch_size), desc="Adding to Chroma"):
            end_idx = min(i + batch_size, len(chunks))
            
            batch_ids = chunk_ids[i:end_idx]
            batch_documents = documents[i:end_idx]
            batch_metadatas = metadatas[i:end_idx]
            batch_embeddings = embeddings[i:end_idx].tolist()
            
            try:
                self.collection.add(
                    ids=batch_ids,
                    documents=batch_documents,
                    metadatas=batch_metadatas,
                    embeddings=batch_embeddings
                )
            except Exception as e:
                print(f"Error adding batch {i}: {str(e)}")
                print(f"Sample metadata: {batch_metadatas[0] if batch_metadatas else 'None'}")
                raise
        
        print(f"Successfully added {len(chunks)} chunks to collection")
        print(f"Total documents in collection: {self.collection.count()}")
    
    def test_retrieval(self, query: str, n_results: int = 5):
        """Test vector similarity search with enhanced metadata display"""
        print(f"\nTesting retrieval with query: '{query}'")
        
        results = self.collection.query(
            query_texts=[query],
            n_results=n_results,
            include=['documents', 'metadatas', 'distances']
        )
        
        print(f"\nTop {n_results} results:")
        for i, (doc, metadata, distance) in enumerate(zip(
            results['documents'][0],
            results['metadatas'][0], 
            results['distances'][0]
        )):
            print(f"\n{i+1}. Distance: {distance:.4f}")
            print(f"   Title: {metadata.get('paper_title', 'Unknown')}")
            print(f"   Authors: {metadata.get('authors', 'Unknown')}")
            print(f"   DOI: {metadata.get('doi', 'N/A')}")
            print(f"   Journal: {metadata.get('journal', 'Unknown')}")
            print(f"   Section: {metadata.get('section', 'Unknown')}")
            print(f"   Document: {metadata.get('document_id', 'Unknown')}")
            print(f"   Content: {doc[:150]}...")
    
    def get_collection_stats(self):
        """Get enhanced collection statistics"""
        count = self.collection.count()
        
        if count > 0:
            sample = self.collection.get(limit=min(100, count), include=['metadatas'])
            
            doc_counts = {}
            section_counts = {}
            journal_counts = {}
            author_counts = {}
            
            for metadata in sample['metadatas']:
                doc_id = metadata.get('document_id', 'Unknown')
                section = metadata.get('section', 'Unknown')
                journal = metadata.get('journal', 'Unknown')
                authors = metadata.get('authors', '')
                
                doc_counts[doc_id] = doc_counts.get(doc_id, 0) + 1
                section_counts[section] = section_counts.get(section, 0) + 1
                if journal and journal != 'Unknown':
                    journal_counts[journal] = journal_counts.get(journal, 0) + 1
                
                # Count individual authors
                if authors:
                    for author in authors.split(', ')[:3]:  # Top 3 authors only
                        if author.strip():
                            author_counts[author.strip()] = author_counts.get(author.strip(), 0) + 1
            
            print(f"\n{'='*60}")
            print("ENHANCED VECTOR DATABASE STATISTICS")
            print(f"{'='*60}")
            print(f"📊 Total chunks: {count}")
            print(f"📊 Unique documents: {len(doc_counts)}")
            print(f"📊 Avg chunks per document: {np.mean(list(doc_counts.values())):.1f}")
            
            print(f"\n📑 Most common sections:")
            for section, count in sorted(section_counts.items(), key=lambda x: x[1], reverse=True)[:5]:
                print(f"   • {section}: {count} chunks")
            
            print(f"\n📚 Most common journals:")
            for journal, count in sorted(journal_counts.items(), key=lambda x: x[1], reverse=True)[:3]:
                print(f"   • {journal}: {count} chunks")
            
            print(f"\n👥 Most frequent authors:")
            for author, count in sorted(author_counts.items(), key=lambda x: x[1], reverse=True)[:5]:
                print(f"   • {author}: {count} chunks")
    
    def validate_metadata_quality(self):
        """Validate the quality of stored metadata"""
        print(f"\n{'='*50}")
        print("METADATA QUALITY VALIDATION")
        print(f"{'='*50}")
        
        sample = self.collection.get(limit=10, include=['metadatas'])
        
        quality_checks = {
            'has_title': 0,
            'has_authors': 0,
            'has_doi': 0,
            'has_journal': 0,
            'has_date': 0,
            'has_source_url': 0
        }
        
        total_samples = len(sample['metadatas'])
        
        for metadata in sample['metadatas']:
            if metadata.get('paper_title') and metadata.get('paper_title') != 'Unknown Title':
                quality_checks['has_title'] += 1
            if metadata.get('authors') and metadata.get('authors') != '':
                quality_checks['has_authors'] += 1
            if metadata.get('doi') and metadata.get('doi') != '':
                quality_checks['has_doi'] += 1
            if metadata.get('journal') and metadata.get('journal') != '':
                quality_checks['has_journal'] += 1
            if metadata.get('publication_date') and metadata.get('publication_date') != '':
                quality_checks['has_date'] += 1
            if metadata.get('source_url') and metadata.get('source_url') != '':
                quality_checks['has_source_url'] += 1
        
        for check, count in quality_checks.items():
            percentage = (count / total_samples) * 100
            print(f"✓ {check.replace('_', ' ').title()}: {percentage:.1f}% ({count}/{total_samples})")

def main():
    """Main execution function with enhanced setup"""
    
    # Use enhanced vector database builder
    vector_db = EnhancedVectorDatabaseBuilder(
        persist_directory="D:/PsyWiz/enhanced_vector_db",  # New DB for enhanced data
        embedding_model="sentence-transformers/all-MiniLM-L6-v2",
        collection_name="psywiz_enhanced_papers"
    )
    
    # Load enhanced chunks
    chunks_file = "D:/PsyWiz/updated_chunks/all_chunks.json"
    chunks = vector_db.load_chunks(chunks_file)
    
    # Add to collection with enhanced metadata
    vector_db.add_chunks_to_collection(chunks, batch_size=50)
    
    # Get enhanced statistics
    vector_db.get_collection_stats()
    
    # Validate metadata quality
    vector_db.validate_metadata_quality()
    
    # Test enhanced retrieval
    test_queries = [
        "depression prevalence in diabetes patients",
        "anxiety symptoms in adolescents", 
        "food insecurity and mental health",
        "stigma toward mental illness"
    ]
    
    for query in test_queries:
        vector_db.test_retrieval(query, n_results=3)
    
    print(f"\n{'='*60}")
    print("🎉 ENHANCED VECTOR DATABASE SETUP COMPLETE!")
    print(f"{'='*60}")
    print(f"📍 Database location: {vector_db.persist_directory}")
    print(f"📊 Collection name: {vector_db.collection_name}")
    print(f"✨ Enhanced metadata ready for RAG pipeline!")
    print(f"\nNow your RAG responses will include:")
    print(f"   • Complete paper titles")
    print(f"   • Full author names")
    print(f"   • DOIs and publication dates")
    print(f"   • Journal names")
    print(f"   • Source URLs")

if __name__ == "__main__":
    main()

c:\Users\Lenovo\anaconda3\envs\psywiz-backend\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Could not load existing collection: Collection psywiz_enhanced_papers does not exist.
Created new collection: psywiz_enhanced_papers
Loading chunks from: D:/PsyWiz/updated_chunks/all_chunks.json
Loaded 4579 chunks
Sample chunk metadata keys: ['document_id', 'chunk_id', 'source_file', 'paper_title', 'authors', 'doi', 'publication_date', 'journal', 'source_url', 'section', 'section_index', 'chunk_index', 'content_preview', 'token_count', 'abstract', 'keywords', 'total_chunks', 'priority_score']
Generating embeddings for 4579 chunks...


Embedding batches: 100%|██████████| 144/144 [09:57<00:00,  4.15s/it]


...


ENHANCED VECTOR DATABASE STATISTICS
📊 Total chunks: 4579
📊 Unique documents: 5
📊 Avg chunks per document: 20.0

📑 Most common sections:
   • Discussion: 11 chunks
   • RESULTS: 7 chunks
   • DISCUSSION: 6 chunks
   • Results: 4 chunks
   • Strengths and limitations: 3 chunks

📚 Most common journals:
   • BMJ Open: 28 chunks
   • PLoS ONE: 21 chunks
   • BMC Pregnancy and Childbirth: 21 chunks

👥 Most frequent authors:
   • Samia Hussain: 28 chunks
   • Safieh Shah: 21 chunks
   • Nafisa Insan: 21 chunks
   • Robin A Richardson: 18 chunks
   • Muhammad Omair Husain: 12 chunks

METADATA QUALITY VALIDATION


Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


✓ Has Title: 100.0% (10/10)
✓ Has Authors: 100.0% (10/10)
✓ Has Doi: 100.0% (10/10)
✓ Has Journal: 100.0% (10/10)
✓ Has Date: 100.0% (10/10)
✓ Has Source Url: 100.0% (10/10)

Testing retrieval with query: 'depression prevalence in diabetes patients'


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



Top 3 results:

1. Distance: 0.4413
   Title: A systematic review and meta-analysis of the prevalence of common mental disorders in people with non-communicable diseases in Bangladesh, India, and Pakistan
   Authors: Eleonora P Uphoff
   DOI: 10.7189/jogh.09.020417
   Journal: Journal of Global Health
   Section: Patients with diabetes
   Document: article_2
   Content: The pooled prevalence of depression from 43 estimates in 41 studies of patients with diabetes is 40% (95% CI = 34 to 45) ( Figure 3 ). Figure 3. Meta-...

2. Distance: 0.5539
   Title: Living with depression and diabetes: A qualitative study in Bangladesh and Pakistan
   Authors: Hannah Maria Jennings
   DOI: 10.1371/journal.pgph.0002846
   Journal: PLOS Global Public Health
   Section: Depression: Experiences, coping, care-seeking and treatment
   Document: article_54
   Content: When asked specifically about diabetes, while several participants did not explicitly identify a link between diabetes and mental health, ma

---
# DB BROWSER

Check what files exist

In [ ]:

import os
from pathlib import Path

db_path = Path("D:/PsyWiz/vector_db")
print("Database files:")
for file in db_path.rglob("*"):
    if file.is_file():
        size_mb = file.stat().st_size / (1024*1024)
        print(f"  {file.name}: {size_mb:.1f} MB")

Database files:
  chroma.sqlite3: 1.6 MB
  data_level0.bin: 16.0 MB
  header.bin: 0.0 MB
  length.bin: 0.0 MB
  link_lists.bin: 0.0 MB


Test persistence - connect to existing DB

In [ ]:
# test_db = VectorDatabaseBuilder()
# print(f"Existing chunks: {test_db.collection.count()}")
# test_db.test_retrieval("depression", n_results=2)

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2
Loaded existing collection: psywiz_papers
Existing chunks: 87

Testing retrieval with query: 'depression'

Top 2 results:

1. Distance: 0.9411
   Section: journal of
   Document: article_2
   Content: health
global
© 2019 The Author(s)
JoGH © 2019 ISoGH
Common mental disorders and non-communicable
diseases
The WHO estimated that 4.4% of the global population was living
with depression and 3.6% was ...

2. Distance: 1.0477
   Section: The mean age of participants was 12.9 ± 1.6 years in boys and 12.0 ±
   Document: article_3
   Content: Specifically, dietary diver-
sity, food insecurity and folate deficiency were significantly associ-
ated with greater depression symptoms in both sexes (respectively,
for boys, IRR = 0.906; CI 95% = 0...
